https://octopus.developers.institute/courses/collection/117/course/659/section/1787/chapter/4493#

Daily Challenge : Preprocess & fine-tune transformer-based models

👩‍🏫 👩🏿‍🏫 What You’ll learn
In this daily challenge, you will learn how to preprocess and fine-tune transformer-based models, specifically BERT and XLM-RoBERTa, for text classification tasks. You will gain an understanding of:

How tokenization works for these models.
How to properly format input data.
How to fine-tune transformer models for classification tasks.
How to perform cross-validation using k-fold splitting.

🛠️ What you will create
By the end of this challenge, you will have a fine-tuned transformer model (BERT or XLM-RoBERTa) capable of classifying text into different categories. Additionally, you will structure the data for training, validate it using cross-validation, and understand how to optimize these models for better performance.

In [1]:
#1. Understanding BERT and XLM-RoBERTa
    # Objective: Learn how transformer models work and their role in NLP tasks.

    # Instructions:
        # Read through the descriptions of BERT and XLM-RoBERTa.
        # Understand how these models process text using tokenization.
        # Learn about different pre-trained versions of these models and their characteristics.



# Read through the descriptions of BERT and XLM-RoBERTa.
    # BERT (Bidirectional Encoder Representations from Transformers) -développé par Google (2018)
        # Modèle bidirectionnel qui lit la phrase à gauche et à droite du mot à prédire.
        # Excellente performance en classification de texte, analyse de sentiments, Questions-réponses.
        # Pré-entraînement sur deux tâches :
            # Masked Language Modeling (MLM) : certains mots sont masqués et le modèle doit les deviner.
        # Next Sentence Prediction (NSP) : prédire si une phrase suit logiquement une autre.

    # XLM-RoBERTa
      # Variante multilingue de RoBERTa (qui est une amélioration de BERT).
      # Entraîné sur 100 langues.
      # Supprime la tâche NSP.
      # Utilise plus de données et un entraînement plus long que BERT → meilleurs résultats multilingues.

# Résumé comparatif
Étape	                BERT	                    XLM-RoBERTa
Type de tokenizer	    WordPiece	                SentencePiece
Vocabulaire	          Entraîné sur l'anglais	  Entraîné sur 100 langues
Tokens spéciaux     	[CLS], [SEP]	            <s>, </s>
Subword format	      ##token	                  ▁token
NSP support	          ✅ Oui	                  ❌ Non

# Understand how these models process text using tokenization.

  # Tokenisation = découper un texte brut en unités exploitables ("tokens") que les modèles transformer peuvent convertir en vecteurs numériques.
    # Word-level → un mot = un token
    # Subword-level (BERT, XLM-R) → un mot = plusieurs sous-tokens si inconnu
    # Character-level → chaque lettre est un token
  # Étapes du traitement d’un texte avec BERT et XLM-RoBERTa
    # 1/Texte brut
      # Exemple :
      text = "Transformers are revolutionizing NLP."
    #2/Tokenisation (sous-mots)
      a) BERT (bert-base-uncased) :
from transformers import BertTokenizer
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
tokens = bert_tokenizer.tokenize(text)
print(tokens)

Sortie : ['transformers', 'are', 're', '##volution', '##izing', 'nlp', '.']
## = indique un sous-token (sous-mot attaché à un mot connu).
BERT découpe les mots inconnus en morceaux connus de son vocabulaire.

      b) XLM-RoBERTa (xlm-roberta-base) :
from transformers import XLMRobertaTokenizer
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")
tokens = xlm_tokenizer.tokenize(text)
print(tokens)

Sortie typique : ['▁Transformers', '▁are', '▁revolutionizing', '▁N', 'L', 'P', '.']
▁ = indique le début d’un mot.
XLM-R utilise un vocabulaire multilingue basé sur SentencePiece.


In [21]:
  # Functions to use:

from transformers import BertTokenizer, XLMRobertaTokenizer

In [5]:
    # CODE pour ZIP
import os, zipfile, glob, pandas as pd

In [10]:
import zipfile
import os

OUTER_ZIP = "Basics of BERT and XLM-RoBERTa - PyTorch - 2.zip"

if os.path.exists(OUTER_ZIP):
    with zipfile.ZipFile(OUTER_ZIP, 'r') as z:
        z.extractall("data")
        print("Fichier extrait dans le dossier 'data'")
else:
    print(f"❌ Fichier non trouvé : {OUTER_ZIP}")

Fichier extrait dans le dossier 'data'


In [9]:
# Lister les fichiers dans le répertoire actuel
print(os.listdir())

from google.colab import files
uploaded = files.upload()

['.config', 'sample_submission.csv', 'sample_data']


Saving Basics of BERT and XLM-RoBERTa - PyTorch - 2.zip to Basics of BERT and XLM-RoBERTa - PyTorch - 2.zip


In [11]:
# :un: unpack the outer zip (already done)
OUTER_ZIP = "Basics of BERT and XLM-RoBERTa - PyTorch - 2.zip"
with zipfile.ZipFile(OUTER_ZIP) as z:
  z.extractall("data")

In [14]:
# :deux: find the folder that was created
root = glob.glob("data/*BERT*XLM*")[0]          # 'data/Basics of BERT and XLM-RoBERTa - PyTorch - 2'
print("root dir →", root, os.listdir(root))

root dir → data/Basics of BERT and XLM-RoBERTa - PyTorch ['sample_submission.csv', 'train.csv.zip', 'test.csv.zip']


In [17]:
# :trois: unzip the inner train/test archives
for inner_zip in glob.glob(f"{root}/*.zip"):
  with zipfile.ZipFile(inner_zip) as z:
        z.extractall(root)                      # drops train.csv / test.csv next to the zips
  print("unzipped:", inner_zip)

unzipped: data/Basics of BERT and XLM-RoBERTa - PyTorch/train.csv.zip
unzipped: data/Basics of BERT and XLM-RoBERTa - PyTorch/test.csv.zip


In [19]:
# :quatre: now load
TRAIN_CSV = f"{root}/train.csv"
TEST_CSV  = f"{root}/test.csv"      # will exist if the dataset shipped one
train_df = pd.read_csv(TRAIN_CSV)
print("\nTrain preview:\n", train_df.head())
print("\nLabel distribution:\n", train_df['label'].value_counts())
test_df = pd.read_csv(TEST_CSV) if os.path.exists(TEST_CSV) else None



Train preview:
            id                                            premise  \
0  5130fd2cb5  and these comments were considered in formulat...   
1  5b72532a0b  These are issues that we wrestle with in pract...   
2  3931fbe82a  Des petites choses comme celles-là font une di...   
3  5622f0c60b  you know they can't really defend themselves l...   
4  86aaa48b45  ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...   

                                          hypothesis lang_abv language  label  
0  The rules developed in the interim were put to...       en  English      0  
1  Practice groups are not permitted to work on t...       en  English      2  
2              J'essayais d'accomplir quelque chose.       fr   French      0  
3  They can't defend themselves because of their ...       en  English      0  
4    เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร       th     Thai      1  

Label distribution:
 label
0    4176
2    4064
1    3880
Name: count, dtype: int64


In [20]:
!pip -q install -U transformers datasets evaluate scikit-learn accelerate --progress-bar off
!pip install tensorflow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.7.1 which is incompatible.


In [ ]:
#2. Tokenizing Text - Objective: Understand how to tokenize text using pre-trained tokenizers.

  # Instructions:
      # Use the BertTokenizer and XLMRobertaTokenizer to convert sentences into tokenized input.
      # Experiment with single-sentence and two-sentence tokenization.
      # Explore the different token types, such as input_ids, attention_mask, and labels.
      # Functions to use:

In [22]:
#  # Use the BertTokenizer and XLMRobertaTokenizer to convert sentences into tokenized input.
from transformers import BertTokenizer, XLMRobertaTokenizer

In [23]:
# Tokenizers loading :
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

In [25]:
# Experiment with single-sentence and two-sentence tokenization.
sentence = "Transformers are revolutionizing NLP."
# Avec BERT :
encoding_bert = bert_tokenizer.encode_plus(
    sentence,
    add_special_tokens=True,
    return_attention_mask=True,
    return_token_type_ids=True,
    return_tensors="pt"   # format PyTorch
)
# Avec XLM-R :
encoding_xlm = xlm_tokenizer.encode_plus(
    sentence,
    add_special_tokens=True,
    return_attention_mask=True,
    return_tensors="pt"
)

In [26]:
# Explore the different token types, such as input_ids, attention_mask, and labels.
    # With BERT :
print("Input IDs:", encoding_bert["input_ids"])
print("Attention Mask:", encoding_bert["attention_mask"])
print("Token Type IDs:", encoding_bert["token_type_ids"])  # 0 = phrase A, 1 = phrase B


Input IDs: tensor([[  101, 19081,  2024,  4329,  6026, 17953,  2361,  1012,   102]])
Attention Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])
Token Type IDs: tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]])


In [27]:
    # With  XLM-R (pas de token_type_ids) :
print("Input IDs:", encoding_xlm["input_ids"])
print("Attention Mask:", encoding_xlm["attention_mask"])

Input IDs: tensor([[    0, 11062, 82772,     7,   621, 98834, 84382,   541, 37352,     5,
             2]])
Attention Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [30]:
# Tokenisation of two sentences :

sentence1 = "The Eiffel Tower is in Paris."
sentence2 = "It is a popular tourist attraction."

  # BERT :
encoding_pair = bert_tokenizer.encode_plus(
    sentence1,
    sentence2,
    add_special_tokens=True,
    return_attention_mask=True,
    return_token_type_ids=True,
    return_tensors="pt"
)
  # XLM-R :
encoding_pair_xlm = xlm_tokenizer.encode_plus(
    sentence1,
    sentence2,
    add_special_tokens=True,
    return_attention_mask=True,
    return_tensors="pt"
)

# Key differences :
    # BERT adds [CLS] sentence1 [SEP] sentence2 [SEP]
    # XLM-R adds <s> sentence1 </s> </s> sentence2 </s>

In [31]:
# Recoding of input_ids
    # As to better catch what input_ids means, let's reconvert them into text :

    # With BERT
print(bert_tokenizer.decode(encoding_bert["input_ids"][0]))

    # With XLM-R
print(xlm_tokenizer.decode(encoding_xlm["input_ids"][0]))

[CLS] transformers are revolutionizing nlp. [SEP]
<s> Transformers are revolutionizing NLP.</s>


In [32]:
# Exemple for tokenisation + visualization :

text = "Artificial intelligence is the future."

tokens = bert_tokenizer.tokenize(text)
ids = bert_tokenizer.convert_tokens_to_ids(tokens)
decoded = bert_tokenizer.decode(ids)

print("Tokens:", tokens)
print("IDs:", ids)
print("Decoded:", decoded)

Tokens: ['artificial', 'intelligence', 'is', 'the', 'future', '.']
IDs: [7976, 4454, 2003, 1996, 2925, 1012]
Decoded: artificial intelligence is the future.


In [33]:
# 3. Preparing Input Data for the Model - Objective: Format input data correctly for transformer models.

    # Instructions:
        # Ensure that input sentences are padded and possibly truncated to max_length.
        # Understand and set special tokens such as <s> and </s>.
        # Learn about attention_mask and how it helps the model ignore padding tokens.
        # Functions to use:
            # tokenizer.encode_plus()         → encode with padding/troncature, etc.
            # tokenizer.special_tokens_map    → list of special tokens ([CLS], [SEP], etc.)
            # tokenizer.vocab_size            → model vocabulary size

In [37]:
# Example : Prepare a sentence
sentence = "Transformers are revolutionizing NLP."
from transformers import BertTokenizer, XLMRobertaTokenizer

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

text = "Deep learning is changing the world."

# Implement encode_plus() with padding + troncature (cf. # Ensure that input sentences are padded and possibly truncated to max_length.)
bert_encoding = bert_tokenizer.encode_plus(
    text,
    add_special_tokens=True,       # ajoute [CLS] et [SEP]
    padding='max_length',          # ajoute des 0 jusqu'à max_length
    truncation=True,               # tronque si trop long
    max_length=12,                 # longueur cible
    return_attention_mask=True,
    return_token_type_ids=True,
    return_tensors="pt"
)

xlm_encoding = xlm_tokenizer.encode_plus(
    text,
    add_special_tokens=True,
    padding='max_length',
    truncation=True,
    max_length=12,
    return_attention_mask=True,
    return_tensors="pt"
)

# Result :
print("Input IDs (BERT):", bert_encoding["input_ids"])
print("Attention Mask (BERT):", bert_encoding["attention_mask"])
print("Token Type IDs (BERT):", bert_encoding["token_type_ids"])

print("Input IDs (XLM-R):", xlm_encoding["input_ids"])
print("Attention Mask (XLM-R):", xlm_encoding["attention_mask"])

Input IDs (BERT): tensor([[ 101, 2784, 4083, 2003, 5278, 1996, 2088, 1012,  102,    0,    0,    0]])
Attention Mask (BERT): tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]])
Token Type IDs (BERT): tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
Input IDs (XLM-R): tensor([[     0,  62723,  52080,     83, 151134,     70,   8999,      5,      2,
              1,      1,      1]])
Attention Mask (XLM-R): tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]])


# Understand and set special tokens such as <s> and </s>.


    # Les tokens spéciaux sont automatiquement ajoutés lors de l’encodage avec add_special_tokens=True. Ils jouent un rôle structurel essentiel pour les modèles transformer.

    # Pourquoi des tokens spéciaux ?
Token	          Rôle	                                               Utilisé par
[CLS] ou <s>	Token de début, sert à classifier ou encoder une phrase entière	BERT, RoBERTa, XLM-R
[SEP] ou </s>	Token de séparation entre deux phrases (phrase A / B)	BERT, XLM-R
[PAD] ou <pad>	Padding (pour égaliser la longueur des séquences)	Tous
[MASK] ou <mask>	Utilisé pendant le Masked Language Modeling (MLM)	Tous
[UNK] ou <unk>	Mot inconnu (non présent dans le vocabulaire)	Tous

    # Accès aux tokens spéciaux dans le tokenizer
from transformers import BertTokenizer, XLMRobertaTokenizer

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

    # ➤ Consulter les tokens spéciaux :
print("BERT special tokens map:")
print(bert_tokenizer.special_tokens_map)

print("\nXLM-RoBERTa special tokens map:")
print(xlm_tokenizer.special_tokens_map)

    # Sortie exemple (BERT) :
{
 'bos_token': '[CLS]',
 'eos_token': '[SEP]',
 'unk_token': '[UNK]',
 'sep_token': '[SEP]',
 'pad_token': '[PAD]',
 'cls_token': '[CLS]',
 'mask_token': '[MASK]'
}
    # Sortie exemple (XLM-R) :
{
 'bos_token': '<s>',
 'eos_token': '</s>',
 'unk_token': '<unk>',
 'sep_token': '</s>',
 'pad_token': '<pad>',
 'cls_token': '<s>',
 'mask_token': '<mask>'
}

In [39]:
# Getting access to special tokens within the tokenizer

from transformers import BertTokenizer, XLMRobertaTokenizer

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

# Consulting special tokens :

print("BERT special tokens map:")
print(bert_tokenizer.special_tokens_map)

print("\nXLM-RoBERTa special tokens map:")
print(xlm_tokenizer.special_tokens_map)

BERT special tokens map:
{'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}

XLM-RoBERTa special tokens map:
{'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}


In [41]:
# Learn about attention_mask and how it helps the model ignore padding tokens.

    # Why using attention_mask for transformers ?
        # Models like BERT or XLM-RoBERTa process fixed-length sequences (e.g., 128 or 512 tokens).
              # To ensure that all inputs have the same length, padding is used ([PAD] or <pad>):
              # Short sentence: "Bonjour" → [101, 7592, 102, 0, 0, 0, 0]
              #                                              ↑ padding
              # But the model should not learn from these 0s → that's why we use the attention_mask.

    # For example :
from transformers import BertTokenizer
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

text = "AI will shape the future."

encoding = bert_tokenizer.encode_plus(
    text,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=10,
    return_attention_mask=True,
    return_tensors="pt"
)

print("Input IDs:", encoding["input_ids"])
print("Attention Mask:", encoding["attention_mask"])

Input IDs: tensor([[ 101, 9932, 2097, 4338, 1996, 2925, 1012,  102,    0,    0]])
Attention Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0]])


In [45]:
# 4. Loading and Exploring the Dataset - Objective: Load the dataset and explore its structure.

  # Instructions:
        # Load the training and testing data from CSV files.
import pandas as pd
        # Display the first few rows to understand its structure.
        # Identify the columns needed for training the model.
        # Functions to use:
TRAIN_CSV = f"{root}/train.csv"
TEST_CSV  = f"{root}/test.csv"      # will exist if the dataset shipped one
train_df = pd.read_csv(TRAIN_CSV)
print("\nTrain preview:\n", train_df.head())
print("\nLabel distribution:\n", train_df['label'].value_counts())
test_df = pd.read_csv(TEST_CSV) if os.path.exists(TEST_CSV) else None


Train preview:
            id                                            premise  \
0  5130fd2cb5  and these comments were considered in formulat...   
1  5b72532a0b  These are issues that we wrestle with in pract...   
2  3931fbe82a  Des petites choses comme celles-là font une di...   
3  5622f0c60b  you know they can't really defend themselves l...   
4  86aaa48b45  ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...   

                                          hypothesis lang_abv language  label  
0  The rules developed in the interim were put to...       en  English      0  
1  Practice groups are not permitted to work on t...       en  English      2  
2              J'essayais d'accomplir quelque chose.       fr   French      0  
3  They can't defend themselves because of their ...       en  English      0  
4    เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร       th     Thai      1  

Label distribution:
 label
0    4176
2    4064
1    3880
Name: count, dtype: int64


In [49]:
        # Display the first few rows to understand its structure.
print("Train set (shape):", train_df.shape)
print("Test set (shape):", test_df.shape)

Train set (shape): (12120, 6)
Test set (shape): (5195, 5)


In [50]:
print("\nTrain preview:")
print(train_df.head())


Train preview:
           id                                            premise  \
0  5130fd2cb5  and these comments were considered in formulat...   
1  5b72532a0b  These are issues that we wrestle with in pract...   
2  3931fbe82a  Des petites choses comme celles-là font une di...   
3  5622f0c60b  you know they can't really defend themselves l...   
4  86aaa48b45  ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...   

                                          hypothesis lang_abv language  label  
0  The rules developed in the interim were put to...       en  English      0  
1  Practice groups are not permitted to work on t...       en  English      2  
2              J'essayais d'accomplir quelque chose.       fr   French      0  
3  They can't defend themselves because of their ...       en  English      0  
4    เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร       th     Thai      1  


In [51]:
print("\nTest preview:")
print(test_df.head())


Test preview:
           id                                            premise  \
0  c6d58c3f69  بکس، کیسی، راہیل، یسعیاہ، کیلی، کیلی، اور کولم...   
1  cefcc82292                             هذا هو ما تم نصحنا به.   
2  e98005252c  et cela est en grande partie dû au fait que le...   
3  58518c10ba                   与城市及其他公民及社区组织代表就IMA的艺术发展进行对话&amp   
4  c32b0d16df                              Она все еще была там.   

                                          hypothesis lang_abv language  
0  کیسی کے لئے کوئی یادگار نہیں ہوگا, کولمین ہائی...       ur     Urdu  
1  عندما يتم إخبارهم بما يجب عليهم فعله ، فشلت ال...       ar   Arabic  
2                             Les mères se droguent.       fr   French  
3                            IMA与其他组织合作，因为它们都依靠共享资金。       zh  Chinese  
4     Мы думали, что она ушла, однако, она осталась.       ru  Russian  


In [ ]:
# 5: Creating Cross-Validation Folds - Objective : Implement k-fold cross-validation using StratifiedKFold to ensure balanced label distribution across folds — crucial for classification tasks.


In [55]:
print(train_df.columns.tolist())

['id', 'premise', 'hypothesis', 'lang_abv', 'language', 'label']


In [71]:
# Import
from sklearn.model_selection import StratifiedKFold

import pandas as pd
import numpy as np

# 1. Définir l'objet StratifiedKFold (avec 2 splits par défaut ici)
kf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

# 2. Exemple avec des données X (features) et y (labels)
# Supposons que X et y viennent d’un DataFrame train_df
X = train_df["premise"]
y = train_df["hypothesis"]

# 3. Utiliser split()
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"Fold {fold + 1}:")
    print(f"  Train indices: {train_idx[:5]} ...")
    print(f"  Validation indices: {val_idx[:5]} ...")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

Fold 1:
  Train indices: [1 3 5 7 9] ...
  Validation indices: [0 2 4 6 8] ...
Fold 2:
  Train indices: [0 2 4 6 8] ...
  Validation indices: [1 3 5 7 9] ...


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:784: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_of_target_y = type_of_target(y)
/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=2.
  warnings.warn(


In [75]:
# Ensure that each fold maintains the same label distribution.
# Store the training and validation splits in separate lists.

from sklearn.model_selection import StratifiedKFold

import pandas as pd
import numpy as np

# Défine StratifiedKFold (with 2 splits here)
kf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

# Initialize lists
train_indices_list = []
val_indices_list = []

# Complete lists with splits
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    train_indices_list.append(train_idx)
    val_indices_list.append(val_idx)

    print(f"Fold {fold + 1} ✅")
    print("  Label distribution in training set:")
    print(y.iloc[train_idx].value_counts(normalize=True).sort_index())

    print("  Label distribution in validation set:")
    print(y.iloc[val_idx].value_counts(normalize=True).sort_index())
    print("-" * 50)

/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:784: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_of_target_y = type_of_target(y)
/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=2.
  warnings.warn(


Fold 1 ✅
  Label distribution in training set:
hypothesis
 "The Man Nobody Knows," by Bruce Barton, was never a best seller.                                        0.000165
 And environmentalists have on occasion attacked religion.                                                0.000165
 IDAs are special in that low-income savers receive matching funds from federal and state governments.    0.000165
 None of the critics like that one                                                                        0.000165
 You have to dig them up every five years, throw them away and then start all over again.                 0.000165
                                                                                                            ...   
邮政密度对成本没有影响。                                                                                              0.000165
门票的销售额还不到这笔钱的一半。                                                                                          0.000165
阿拉贝拉刚好位于穿过港口边缘的地方，那是城市

# Identify the columns needed for training the model.

# Element to identify	                  Why
    # The "text" column	            Raw text to be processed by the model
    # The "label" column	          Target class (often 0 / 1)
    # Other columns	                Can be removed or ignored

Daily Challenge : Preprocess & fine-tune transformer-based models
14 h 20
CODE pour ZIP
import os, zipfile, glob, pandas as pd
# :un: unpack the outer zip (already done)
OUTER_ZIP = "Basics of BERT and XLM-RoBERTa - PyTorch - 2.zip"
with zipfile.ZipFile(OUTER_ZIP) as z:
    z.extractall("data")
# :deux: find the folder that was created
root = glob.glob("data/*BERT*XLM*")[0]          # 'data/Basics of BERT and XLM-RoBERTa - PyTorch - 2'
print("root dir →", root, os.listdir(root))
# :trois: unzip the inner train/test archives
for inner_zip in glob.glob(f"{root}/*.zip"):
    with zipfile.ZipFile(inner_zip) as z:
        z.extractall(root)                      # drops train.csv / test.csv next to the zips
    print("unzipped:", inner_zip)
# :quatre: now load
TRAIN_CSV = f"{root}/train.csv"
TEST_CSV  = f"{root}/test.csv"      # will exist if the dataset shipped one
train_df = pd.read_csv(TRAIN_CSV)
print("\nTrain preview:\n", train_df.head())
print("\nLabel distribution:\n", train_df['label'].value_counts())
test_df = pd.read_csv(TEST_CSV) if os.path.exists(TEST_CSV) else None
14 h 21
:cercle_rouge: !pip -q install -U transformers datasets evaluate scikit-learn accelerate --progress-bar off
!pip install tensorflow